In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import optuna
import os
import shap
import warnings
warnings.filterwarnings('ignore')

# 1. 4번 모델 기준 데이터 세트로 롤백 (신규 추가된 다중 범주 결합 폐기)
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

dup_cols = [col for col in train.columns if col != 'ID']
train = train.drop_duplicates(subset=dup_cols).reset_index(drop=True)

train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)

    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)

    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)

    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']

    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)

    return data

train = add_features(train)
test = add_features(test)

cat_cols = ['gender', 'activity', 'smoke_status', 'medical_history',
            'family_medical_history', 'sleep_pattern', 'edu_level']

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

# 2. SHAP 분석을 적용하여 기여도가 낮은 하위 파생 변수 강제 소거
print("★ SHAP 분석 모델 학습 중 ★")
init_model = lgb.LGBMRegressor(random_state=42, n_estimators=300, verbose=-1)
init_model.fit(x_train, y_train, categorical_feature=cat_cols)

explainer = shap.TreeExplainer(init_model)
shap_values = explainer.shap_values(x_train)

shap_sum = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame([x_train.columns.tolist(), shap_sum.tolist()]).T
importance_df.columns = ['feature', 'shap_importance']
importance_df = importance_df.sort_values('shap_importance', ascending=False)

# SHAP 기여도 하위 5개 변수 강제 소거
drop_features = importance_df.tail(5)['feature'].tolist()
print(f"★ SHAP 기반 강제 소거 피처: {drop_features}")

x_train_selected = x_train.drop(columns=drop_features)
x_test_selected = x_test.drop(columns=drop_features)
cat_cols_selected = [col for col in cat_cols if col not in drop_features]

# 3. 선별된 피처를 바탕으로 Optuna 튜닝 및 최종 모델 학습
def objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255, step=16),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 1.0, log=True),
    }
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_mae = []
    
    for tr_idx, val_idx in kf.split(x_train_selected):
        X_tr, X_val = x_train_selected.iloc[tr_idx], x_train_selected.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, categorical_feature=cat_cols_selected)
        
        pred = model.predict(X_val)
        cv_mae.append(mean_absolute_error(y_val, pred))
        
    return np.mean(cv_mae)

print("★ Optuna 튜닝 시작 (30회 반복) ★")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print(f"\n★ Best Optuna CV MAE: {study.best_value:.4f}")

best_params = study.best_params
best_params.update({'objective': 'regression_l1', 'metric': 'mae', 'random_state': 42, 'verbose': -1})

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train_selected))
test_preds = np.zeros(len(x_test_selected))

for tr_idx, val_idx in kf.split(x_train_selected):
    X_tr, X_val = x_train_selected.iloc[tr_idx], x_train_selected.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(X_tr, y_tr, categorical_feature=cat_cols_selected)
    
    oof_preds[val_idx] = final_model.predict(X_val)
    test_preds += final_model.predict(x_test_selected) / kf.n_splits

final_cv_mae = mean_absolute_error(y_train, oof_preds)
print(f"\n★ 최종 모델 자체 점수 (OOF CV MAE): {final_cv_mae:.4f}")

test_preds = np.clip(test_preds, 0, 1)
sample_submission['stress_score'] = test_preds
submit_path = '../submissions/submit_16_rollback_shap.csv'
os.makedirs('../submissions', exist_ok=True)
sample_submission.to_csv(submit_path, index=False)
print(f"★ 제출 파일 생성 완료: {submit_path}")